<a href="https://colab.research.google.com/github/Md-Zeeshan777/FSD/blob/main/CyberSecurityDoubtChatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets torch

In [ ]:
%pip install langchain langchain-community langchain-huggingface \
sentence-transformers faiss-cpu transformers pypdf gradio

INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.6/330.6 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninst

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("/content/how_hackers_hack_systems_full_pages.pdf")
documents = loader.load()

In [ ]:
!pip install -U langchain langchain-community langchain-huggingface faiss-cpu transformers sentence-transformers pypdf accelerate


  Using cached transformers-5.1.0-py3-none-any.whl.metadata (31 kB)
INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.1/500.1 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.1/158.1 kB 11.8 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.9
    Uninstalling langchain-core-1.2.9:
      Successfully uninstalled langchain-core-1.2.9
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.0.7
    Uninstalling langgraph-1.0.7:
      Successfully uninstalled langgraph-1.0.7
  Attempting uninstall: langchain
    Found existing installation: langchain 1.2.8
    Uninstalling langchain-1.2.8:
      Successfully uninstalled langchain-1.2.8


In [ ]:
import os

# ---------------- CONFIG ----------------

PDF_PATH = "/content/how_hackers_hack_systems_full_pages.pdf"

CHUNK_SIZE = 500
CHUNK_OVERLAP = 80
TOP_K = 4

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL = "google/flan-t5-base"


# ---------------- LOAD PDF ----------------

from langchain_community.document_loaders import PyPDFLoader


def load_pdf(path):

    if not os.path.exists(path):
        raise FileNotFoundError(f"PDF not found: {path}")

    loader = PyPDFLoader(path)
    return loader.load()


# ---------------- SPLIT ----------------

from langchain_text_splitters import RecursiveCharacterTextSplitter


def split_docs(documents):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP
    )

    return splitter.split_documents(documents)


# ---------------- VECTOR DB ----------------

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS


def create_vectorstore(docs):

    embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL
    )

    return FAISS.from_documents(docs, embeddings)


# ---------------- LLM ----------------

from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline


def load_llm():

    pipe = pipeline(
        "text2text-generation",
        model=LLM_MODEL,
        max_new_tokens=250,
        temperature=0.3
    )

    return HuggingFacePipeline(pipeline=pipe)


# ---------------- RAG ----------------

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


def build_rag_chain(retriever, llm):

    prompt = ChatPromptTemplate.from_template(
        """
You are a helpful AI assistant.

Use ONLY the given context.
If missing, say "I don't know".

Context:
{context}

Question:
{question}

Answer:
"""
    )

    return (
        {
            "context": retriever,
            "question": RunnablePassthrough()
        }
        | prompt
        | llm
        | StrOutputParser()
    )


# ---------------- RUN PIPELINE ----------------

print("📄 Loading PDF...")
documents = load_pdf(PDF_PATH)

print("✂️ Splitting...")
docs = split_docs(documents)
print("Chunks:", len(docs))

print("🧠 Creating embeddings...")
vectorstore = create_vectorstore(docs)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": TOP_K}
)

print("🤖 Loading LLM...")
llm = load_llm()

print("🔗 Building RAG...")
rag_chain = build_rag_chain(retriever, llm)

print("✅ Ready!")


# ---------------- ASK QUESTIONS ----------------

def ask(question):

    return rag_chain.invoke(question)


# Example Test
print("\nTest Answer:")
print(ask("What is ransomware?"))




📄 Loading PDF...
✂️ Splitting...
Chunks: 68
🧠 Creating embeddings...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Token indices sequence length is longer than the specified maximum sequence length for this model (1459 > 512). Running this sequence through the model will result in indexing errors


🔗 Building RAG...
✅ Ready!

Test Answer:
Locking files and demanding payment to restore access


In [ ]:
response = rag_chain.invoke("What is hacking?")
print(response)

gaining unauthorized access to computer systems or networks


In [ ]:
response = rag_chain.invoke("Why are public Wi-Fi networks risky?")
print(response)

weaker protections


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
#Load PDF
loader = PyPDFLoader("/content/how_hackers_hack_systems_full_pages.pdf")
documents = loader.load()

documents[:1]

#Split text into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30
)

docs = text_splitter.split_documents(documents)
print("Number of chunks:", len(docs))

#Create embeddings
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

#Store embeddings in FAISS
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(docs, embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

#Load LLM (CORRECT pipeline for FLAN-T5)
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

pipe = pipeline(
    "text2text-generation",   # ✅ correct for T5
    model="google/flan-t5-base",
    max_new_tokens=200
)

llm = HuggingFacePipeline(pipeline=pipe)

#Build RAG chain (MODERN LCEL – no deprecated APIs)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    """
    Answer the question in detail using ONLY the context below.
    If the answer is not in the context, say "I don't know".

    Context:
    {context}

    Question:
    {question}
    """
)

rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
import gradio as gr


def chat(message, history):
    response = rag_chain.invoke(message)
    history.append((message, response))
    return history, ""


def toggle_button(text):
    # Enable button only if text is not empty
    if text.strip() == "":
        return gr.update(interactive=False)
    else:
        return gr.update(interactive=True)


with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 📘 PDF RAG Chatbot
    Ask questions from your PDF using AI
    """)

    chatbot = gr.Chatbot(
        height=450,
        bubble_full_width=False
    )

    msg = gr.Textbox(
        placeholder="Type your question...",
        show_label=False,
        lines=2
    )

    with gr.Row():
        send = gr.Button("🚀 Send", variant="primary", interactive=False)
        clear = gr.Button("🗑 Clear")

    state = gr.State([])

    status = gr.Markdown("🟢 Ready")

    # Enable / Disable Send button while typing
    msg.change(
        toggle_button,
        inputs=msg,
        outputs=send
    )

    # Send message
    send.click(
        chat,
        inputs=[msg, state],
        outputs=[chatbot, msg]
    )

    msg.submit(
        chat,
        inputs=[msg, state],
        outputs=[chatbot, msg]
    )

    # Clear chat
    clear.click(
        lambda: ([], []),
        None,
        [chatbot, state]
    )


demo.launch(debug=True)



/tmp/ipython-input-2806648968.py:18: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:
/tmp/ipython-input-2806648968.py:25: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipython-input-2806648968.py:25: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.
  chatbot = gr.Chatbot(
/tmp/ipython-input-2806648968.py:25: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://16979188e25654e4d9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
%%writefile app.py
# =========================
# RAG PDF Chatbot - app.py
# =========================

import gradio as gr

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS

from transformers import pipeline
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


# --------- Load PDF ---------
PDF_PATH = "how_hackers_hack_systems.pdf"

loader = PyPDFLoader(PDF_PATH)
documents = loader.load()


# --------- Split Text ---------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30
)
docs = text_splitter.split_documents(documents)


for doc in documents:
    doc.page_content = doc.page_content.lower()


# --------- Embeddings ---------
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


# --------- Vector Store ---------
vectorstore = FAISS.from_documents(docs, embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})


# --------- LLM (FLAN-T5) ---------
pipe = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    max_new_tokens=200
)
llm = HuggingFacePipeline(pipeline=pipe)


# --------- Prompt ---------
prompt = ChatPromptTemplate.from_template(
    """
    Answer the question using ONLY the context below.
    If the answer is not in the context, say "I don't know".

    Context:
    {context}

    Question:
    {question}
    """
)


# --------- RAG Chain ---------
rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)


# --------- Gradio UI ---------
def chat(question):
    return rag_chain.invoke(question)


demo = gr.Interface(
    fn=chat,
    inputs=gr.Textbox(lines=2, placeholder="Ask from the PDF..."),
    outputs="text",
    title="📚 Project Chatbot",
    description="Ask questions grounded in your PDF using RAG"
)

demo.launch()


Writing app.py


In [ ]:
%%writefile requirements.txt
gradio
langchain
langchain-community
langchain-huggingface
langchain-text-splitters
faiss-cpu
pypdf
transformers
sentence-transformers
torch

Writing requirements.txt
